In [1]:
# from mineru import MinerU
# # 初始化客户端
# client = MinerU()
# #调用api
# result = client.flash_extract("resources/example.pdf")
# #转markdown
# result.save_markdown("r2.md")


In [2]:
# from mineru import MinerU
# from app.core.config import settings
#
# # 创建客户端，Precision 模式，需要Token
# client = MinerU(settings.rag.mineru_token)
# # 解析文件
# result = client.extract("./resources/example.pdf")
#
# # 输出到本地
# result.save_markdown("r8.md", with_images=True)


In [3]:

# 准备一段较长的文本
doc = """
大型语言模型是近年来人工智能领域最令人瞩目的突破性技术之一。GPT-3、GPT-4、BERT、T5等模型的出现，彻底改变了自然语言处理领域的研究范式。

这些模型基于深度学习架构，通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

检索增强生成是近年来兴起的一种重要技术范式，它通过将大型语言模型与外部知识检索系统相结合，有效解决了模型知识更新困难、事实性错误、缺乏专业领域知识等问题。在这种范式中，当用户提出一个问题时，系统首先从外部知识库中检索相关的文档片段，然后将这些片段与用户问题一起作为输入传递给大型语言模型进行答案生成。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性。文本分割是构建检索增强生成系统的关键预处理步骤，因为检索的对象通常是文本片段，而分割的质量直接影响后续检索的精确度和召回率。如果分割策略不合理，会带来一系列严重的问题，比如当分割过于粗粒度时检索到的内容可能包含大量无关信息，当分割过于细粒度时则可能丢失关键的上下文信息。

LangChain提供了多种文本分割器实现。CharacterTextSplitter使用指定的单个分隔符对文本进行分割，实现简单高效。RecursiveCharacterTextSplitter则持有一个分隔符优先级列表，能够递归地尝试不同粒度的分隔符。这个分割器因此在处理结构复杂、格式多样的文本时表现更佳，也是LangChain官方推荐的默认分割器。在实际的工程实践中，选择合适的文本分割策略需要综合考虑文本本身的特征和下游任务的具体需求。

通过合理的实验设计和评估，开发者可以找到最适合特定场景的分割策略。
"""

from mineru import MinerU

from langchain_text_splitters import RecursiveCharacterTextSplitter



In [4]:
#按照Token切分
from langchain_text_splitters import CharacterTextSplitter
token_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",    # token分词器编码名
    chunk_size=200,                # 每块最多200 token
    chunk_overlap=40,              # 块之间重叠40 token
)
# 切分文本
chunks = token_splitter.split_text(doc)

print(f"原始文本长度: {len(doc)} token")
print(f"切分为 {len(chunks)} 个块:")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)}token) ---")
    print(chunk)
    print()



Created a chunk of size 337, which is longer than the specified 200


原始文本长度: 706 token
切分为 4 个块:
--- Chunk 1 (123token) ---
大型语言模型是近年来人工智能领域最令人瞩目的突破性技术之一。GPT-3、GPT-4、BERT、T5等模型的出现，彻底改变了自然语言处理领域的研究范式。

这些模型基于深度学习架构，通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

--- Chunk 2 (319token) ---
检索增强生成是近年来兴起的一种重要技术范式，它通过将大型语言模型与外部知识检索系统相结合，有效解决了模型知识更新困难、事实性错误、缺乏专业领域知识等问题。在这种范式中，当用户提出一个问题时，系统首先从外部知识库中检索相关的文档片段，然后将这些片段与用户问题一起作为输入传递给大型语言模型进行答案生成。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性。文本分割是构建检索增强生成系统的关键预处理步骤，因为检索的对象通常是文本片段，而分割的质量直接影响后续检索的精确度和召回率。如果分割策略不合理，会带来一系列严重的问题，比如当分割过于粗粒度时检索到的内容可能包含大量无关信息，当分割过于细粒度时则可能丢失关键的上下文信息。

--- Chunk 3 (223token) ---
LangChain提供了多种文本分割器实现。CharacterTextSplitter使用指定的单个分隔符对文本进行分割，实现简单高效。RecursiveCharacterTextSplitter则持有一个分隔符优先级列表，能够递归地尝试不同粒度的分隔符。这个分割器因此在处理结构复杂、格式多样的文本时表现更佳，也是LangChain官方推荐的默认分割器。在实际的工程实践中，选择合适的文本分割策略需要综合考虑文本本身的特征和下游任务的具体需求。

--- Chunk 4 (33token) ---
通过合理的实验设计和评估，开发者可以找到最适合特定场景的分割策略。



In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 创建递归切分器
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    separators=["\n\n", "\n", "。"]  # 优先级从高到低
)

# 切分文档
chunks = recursive_splitter.split_text(doc)

print(f"切分为 {len(chunks)} 个块:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)}字符) ---")
    print(chunk)
    print()

切分为 6 个块:

--- Chunk 1 (123字符) ---
大型语言模型是近年来人工智能领域最令人瞩目的突破性技术之一。GPT-3、GPT-4、BERT、T5等模型的出现，彻底改变了自然语言处理领域的研究范式。

这些模型基于深度学习架构，通过在海量文本数据上进行预训练，学习到了丰富的语言知识和世界知识。

--- Chunk 2 (183字符) ---
检索增强生成是近年来兴起的一种重要技术范式，它通过将大型语言模型与外部知识检索系统相结合，有效解决了模型知识更新困难、事实性错误、缺乏专业领域知识等问题。在这种范式中，当用户提出一个问题时，系统首先从外部知识库中检索相关的文档片段，然后将这些片段与用户问题一起作为输入传递给大型语言模型进行答案生成。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性

--- Chunk 3 (170字符) ---
。这种方法不仅提高了模型回答的准确性，还增强了模型的可解释性和可控性。文本分割是构建检索增强生成系统的关键预处理步骤，因为检索的对象通常是文本片段，而分割的质量直接影响后续检索的精确度和召回率。如果分割策略不合理，会带来一系列严重的问题，比如当分割过于粗粒度时检索到的内容可能包含大量无关信息，当分割过于细粒度时则可能丢失关键的上下文信息。

--- Chunk 4 (177字符) ---
LangChain提供了多种文本分割器实现。CharacterTextSplitter使用指定的单个分隔符对文本进行分割，实现简单高效。RecursiveCharacterTextSplitter则持有一个分隔符优先级列表，能够递归地尝试不同粒度的分隔符。这个分割器因此在处理结构复杂、格式多样的文本时表现更佳，也是LangChain官方推荐的默认分割器

--- Chunk 5 (46字符) ---
。在实际的工程实践中，选择合适的文本分割策略需要综合考虑文本本身的特征和下游任务的具体需求。

--- Chunk 6 (33字符) ---
通过合理的实验设计和评估，开发者可以找到最适合特定场景的分割策略。



In [6]:
#结构感知切分
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 准备一段较长的文本
with open('./resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()

# 切分依据，这里是按照三级标题
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# 创建切分器
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 文本中是否携带标题
)

# 切分文档，接收str，返回值是list[Document]
chunks = markdown_splitter.split_text(doc)

for chunk in chunks:
    print(chunk.model_dump_json(indent=2))

{
  "id": null,
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（一）教育学萌芽时期代表作"
  },
  "page_content": "# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。",
  "type": "Document"
}
{
  "id": null,
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（二）《学记》主要思想"
  },
  "page_content": "### （二）《学记》主要思想  \n**教育作用**：化民成俗，其必由学；建国君民，教学为先。  \n**教学相长**：教和学两方面互相影响和促进，都得到提高。  \n**豫时孙摩**：包括预防性原则、及时性原则、循序渐进原则、集体教育原则。  \n**长善救失**：“学者有四失，教者必知之。人之学也，或失则多，或失则寡，或失则易，或失则止。此四者，心之莫同也。知其心，然后能救其失也，教也者，长善而救其失者也。”  \n**启发诱导**：道而弗牵，强而弗抑，开而弗达。",
  "type": "Document"
}
{
  "id": null,
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想  \n**教育作用**：庶、富、教；性相近，习相远。  \n**教育对象**：“有教无类”，教育民主思想。  \n**教育目的**：以完善人格为教育的首要目的，培养士和君子。  \n**教学内容**：文、行、忠、义。  \n**教学过程**：学、思、习、

In [7]:
from langchain_community.embeddings import DashScopeEmbeddings
from dotenv import load_dotenv

load_dotenv()

# 初始化DashScopeEmbeddings，只需要指定模型名，会自动取.env读取DASHSCOPE_API_KEY
embeddings = DashScopeEmbeddings(
    model="text-embedding-v4"
)


# 向量化单条文本
text = "我爱上班"
# 向量化后返回的是 list[float]
vector = embeddings.embed_query(text)

print(f"文本: {text}")
print(f"向量维度: {len(vector)}")
print(f"向量前5维: {vector[:5]}")


# 批量向量化
texts = ["我要躺平", "我爱工作", "拒绝加班"]
# 批量向量化，可以理解成处理很多个知识片段，返回 list[list[float]]
vectors = embeddings.embed_documents(texts)

print(f"\n批量向量化: {len(vectors)} 条, 维度: {len(vectors[0])}")


print('==============衡量相似度===============')
import numpy as np


def cosine_similarity(vec1, vec2):
    """余弦相似度函数"""
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# 比较向量相似度
for i, v in enumerate(vectors):
    similarity = cosine_similarity(vector, v)
    print(f"{texts[i]}:", similarity)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_27228\2821360314.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import DashScopeEmbeddings


文本: 我爱上班
向量维度: 1024
向量前5维: [-0.04190880432724953, -0.09621354192495346, 0.036159202456474304, -0.06176867336034775, 0.015297109261155128]

批量向量化: 3 条, 维度: 1024
==============衡量相似度===============
我要躺平: 0.40120064431301805
我爱工作: 0.8227559840016682
拒绝加班: 0.41688926472977106


In [8]:
from langchain_community.embeddings import DashScopeEmbeddings
from dotenv import load_dotenv

load_dotenv()
#初始化向量模型
embeddings = DashScopeEmbeddings(
    model="text-embedding-v4"
)
#对文本进行向量化
text = "我爱上班"
text_list= ["我要躺平", "我爱工作", "拒绝加班"]
vector = embeddings.embed_query(text)
vectors = embeddings.embed_documents(text_list)

#计算相似度
import numpy as np
def cosine_similarity(vec1, vec2):
    """余弦相似度函数"""
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

for i, v in enumerate(vectors):
    similarity = cosine_similarity(vector, v)
    print(f"{text_list[i]}:", similarity)





我要躺平: 0.40120064431301805
我爱工作: 0.8227559840016682
拒绝加班: 0.41688926472977106


In [9]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

# 0.准备一段较长的文本
with open('./resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()
# 1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

# 2.初始化向量数据库
vector_store = InMemoryVectorStore(embedding=ollama_embeddings)
# 保存文档到向量数据库
ids = vector_store.add_documents()
# 3.向量数据库中添加向量
vector_store.add_vectors(vectors, [text_list])




TypeError: InMemoryVectorStore.add_documents() missing 1 required positional argument: 'documents'

In [ ]:
# 在线阶段
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

# 1.用户提问
query = "论语中教育的目的"

# 2.根据问题检索知识片段
documents = vector_store.similarity_search(query)

# 3.把知识片段与问题拼接，形成提示词
# 3.1.获取知识片段上下文
context = "\n".join([doc.page_content for doc in documents])
# 3.2.拼接提示词
prompt = f"""
你是一个教资考试专家，你需要根据提供的上下文回答用户的问题，不能自己编造答案。答案要简短，直接引用原文。
context: ```{context}```
query: ```{query}```
"""

# 4.调用模型，生成答案
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body = {"thinking": {"type": "disabled"}}
)

response = model.invoke(prompt)
print(response.text)

In [10]:
from langchain_community.embeddings import DashScopeEmbeddings
from dotenv import load_dotenv

load_dotenv()
# print(os.getenv("DASHSCOPE_API_KEY"))
# print(os.getenv("DASHSCOPE_BASE_URL"))

# 1.初始化向量模型对象
client= DashScopeEmbeddings(
    model="text-embedding-v4"
)
# 2.文本向量化
text = "我爱上班"
vector1 = client.embed_query(text)

texts = ["我要躺平", "我爱工作", "拒绝加班"]
vestors = client.embed_documents(texts)

# 3.分别计算vector1 和其他三个向量的相似度分数的算法
print('==============衡量相似度===============')
import numpy as np


def cosine_similarity(vec1, vec2):
    """余弦相似度函数"""
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# 4.遍历计算相似度分数
for i, v in enumerate(vestors):
    similarity_score = cosine_similarity(vector1,v)
    print(f"{texts[i]}:", similarity_score)

==============衡量相似度===============
我要躺平: 0.40120064431301805
我爱工作: 0.8227559840016682
拒绝加班: 0.41688926472977106


In [12]:
def rag_query(query:str) -> str:
    # 2.根据问题检索知识片段
    documents = vector_store.similarity_search(query)

    # 3.把知识片段与问题拼接，形成提示词
    # 3.1.获取知识片段上下文
    list_docs = [doc.page_content for doc in documents]
    context = "\n".join(list_docs)
    # 3.2.拼接提示词
    prompt = f"""
    你是一个教资考试专家，你需要根据提供的上下文回答用户的问题，不能自己编造答案。答案要简短，直接引用原文。
    context: ```{context}```
    query: ```{query}```
    """
    # 4.调用模型，生成答案
    model = init_chat_model(
        "deepseek-v4-flash",
        extra_body = {"thinking": {"type": "disabled"}}
    )

    response = model.invoke(prompt)
    return response.text

# rag_query("论语中教育的目的?")
rag_query("你好")

'你好！请问有什么教资考试相关的问题我可以帮你解答？'

In [13]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from app.core.config import settings

# 1.初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}}
)

# 2.构建工具，让agent使用
@tool
def rag_retrieve(query:str) -> str:
    """检索知识库，获取知识片段"""
    result = rag_query(query)
    return result


# 3.初始化Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=[rag_retrieve],
    system_prompt='''
    你是一个教资考试专家，你调用工具检索文档回答用户问题，文档中没有的不要编造，直接说不知道。
    回答直接引用文档中原文，不要自己发挥！
    '''
)
# 4.调用Agent
# query = "你好"
# query = "1+1=？"
query = "论语中教育的目的是什么？"
response = agent.invoke({'messages': [HumanMessage(content=query)]})

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

论语中教育的目的是什么？
================================== Ai Message ==================================
Tool Calls:
  rag_retrieve (call_00_TxiorpEXSF9KL6Ml8IXz3844)
 Call ID: call_00_TxiorpEXSF9KL6Ml8IXz3844
  Args:
    query: 论语 教育目的
================================= Tool Message =================================
Name: rag_retrieve

根据提供的上下文，无法回答该问题。
================================== Ai Message ==================================

根据检索到的文档内容，无法回答该问题。我不知道《论语》中教育的目的是什么。
